In [1]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd

# Identify the project root and locate the raw dataset.
current_folder = Path.cwd().resolve()

possible_data_paths = [
    current_folder / "data" / "raw" / "2020-Apr.csv",           # Running from project_folder/
    current_folder.parent / "data" / "raw" / "2020-Apr.csv"     # Running from project_folder/notebooks/
]

DATA_PATH = next((path for path in possible_data_paths if path.exists()), None)

if DATA_PATH is None:
    searched_paths = "\n".join(str(path) for path in possible_data_paths)
    raise FileNotFoundError(
        "Dataset file was not found. Place '2020-Apr.csv' inside:\n"
        "project_folder/data/raw/\n\n"
        f"Paths checked:\n{searched_paths}"
    )

# Process the file in chunks to avoid loading the full dataset into memory.
CHUNK_SIZE = 250_000

# Select approximately 1% of sessions using a reproducible hash rule.
HASH_MODULUS = 100
HASH_BUCKET = 0

print(f"Working directory: {current_folder}")
print(f"Dataset found: {DATA_PATH}")

Working directory: C:\Users\User\OneDrive\Documents\GACAD_DOMINIC_Capstone\project_folder\notebooks
Dataset found: C:\Users\User\OneDrive\Documents\GACAD_DOMINIC_Capstone\project_folder\data\raw\2020-Apr.csv


In [2]:
required_columns = ["event_type", "user_session"]

event_type_counts = Counter()
sampled_session_events = []
total_rows_processed = 0

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        usecols=required_columns,
        dtype={
            "event_type": "category",
            "user_session": "string"
        },
        chunksize=CHUNK_SIZE
    ),
    start=1
):
    total_rows_processed += len(chunk)

    # Count event types across the full dataset.
    event_type_counts.update(chunk["event_type"].value_counts(dropna=False).to_dict())

    # Keep approximately 1% of sessions using a repeatable hash-based rule.
    session_hash = pd.util.hash_pandas_object(
        chunk["user_session"],
        index=False
    )

    selected_rows = chunk.loc[
        (session_hash % HASH_MODULUS) == HASH_BUCKET,
        ["user_session", "event_type"]
    ]

    if not selected_rows.empty:
        sampled_session_events.append(selected_rows)

    if chunk_number % 20 == 0:
        print(f"Processed {total_rows_processed:,} rows...")

print(f"\nTotal rows processed: {total_rows_processed:,}")

event_type_summary = (
    pd.DataFrame.from_dict(event_type_counts, orient="index", columns=["count"])
    .rename_axis("event_type")
    .reset_index()
)

event_type_summary["percentage"] = (
    event_type_summary["count"] / event_type_summary["count"].sum() * 100
)

event_type_summary = event_type_summary.sort_values(
    by="count",
    ascending=False
).reset_index(drop=True)

display(event_type_summary)

Processed 5,000,000 rows...
Processed 10,000,000 rows...
Processed 15,000,000 rows...
Processed 20,000,000 rows...
Processed 25,000,000 rows...
Processed 30,000,000 rows...
Processed 35,000,000 rows...
Processed 40,000,000 rows...
Processed 45,000,000 rows...
Processed 50,000,000 rows...
Processed 55,000,000 rows...
Processed 60,000,000 rows...
Processed 65,000,000 rows...

Total rows processed: 66,589,268


,event_type,count,percentage
0,view,62353909,93.639577
1,cart,3268600,4.908599
2,purchase,966759,1.451824


In [3]:
if not sampled_session_events:
    raise ValueError("No sampled session records were collected.")

session_sample = pd.concat(sampled_session_events, ignore_index=True)

session_conversion_summary = (
    session_sample
    .assign(
        purchase_flag=lambda df: (
            df["event_type"].astype("string").eq("purchase").astype("int8")
        )
    )
    .groupby("user_session", as_index=False)
    .agg(
        events_in_session=("event_type", "size"),
        session_converted=("purchase_flag", "max")
    )
)

sampled_conversion_rate = session_conversion_summary["session_converted"].mean()

print(f"Sampled sessions: {len(session_conversion_summary):,}")
print(f"Sampled session conversion rate: {sampled_conversion_rate:.2%}")

display(
    session_conversion_summary["session_converted"]
    .value_counts()
    .rename(index={0: "No Purchase", 1: "Purchase"})
    .to_frame("number_of_sessions")
)

Sampled sessions: 116,269
Sampled session conversion rate: 6.70%


,number_of_sessions
session_converted,
No Purchase,108484
Purchase,7785


# Step 1: Problem Understanding and Framing

## Business Problem

The eCommerce dataset contains customer interaction events, including product views, cart additions, and purchases. Most customer activity consists of browsing, while only a smaller portion of sessions eventually lead to a purchase.

The business challenge is to identify customer sessions that are likely to convert while users are still browsing. If high-intent sessions can be detected early, the business can prioritize timely and relevant interventions such as product reminders, cart-recovery prompts, customer-support prompts, delivery incentives, or carefully targeted promotions.

The objective is to focus resources on high-intent sessions instead of applying the same intervention to every visitor.

## Data Science Problem

This project will develop a supervised binary-classification model that predicts whether a user session will result in a future purchase after the first two observed session events.

The unit of analysis is one `user_session`.

The target variable is defined as:

- `future_purchase = 1` when a user session contains a purchase event after the first two observed events.
- `future_purchase = 0` when no purchase event occurs after the first two observed events.

Only the first two events of each eligible session are used to create the model features. This makes the prediction scenario realistic because the model only uses information that would be available early in the session.

Based on the final leakage-safe modeling dataset, there are 92,799 eligible sessions, of which 11,675 resulted in a future purchase. This is equivalent to a future-purchase rate of 12.58%.

## Task Type

This is a supervised binary-classification problem.

The model predicts whether each eligible customer session belongs to one of two classes:

- Future purchase session
- No future purchase session

## Target Metric

The primary model evaluation metric is Precision-Recall Area Under the Curve, or PR-AUC. This is appropriate because future-purchase sessions are less common than non-purchase sessions, making accuracy alone potentially misleading.

Secondary metrics include:

- Precision
- Recall
- F1-score
- ROC-AUC
- Confusion matrix
- Precision among top-ranked or targeted sessions

## Data Leakage Consideration

The model must use only information that would be available at the selected prediction point. In this project, the prediction point is after the first two events of a session.

Purchase events and any customer behavior occurring after the second event are not used as input features. They are used only to define the target variable. This avoids data leakage and ensures that the model reflects a realistic early-session purchase-propensity prediction scenario.

## Business Success Measures

The practical value of the model may be assessed using the following business measures:

- Conversion rate among sessions selected by the model
- Precision among the top-ranked high-intent sessions
- Incremental purchases or sales value resulting from targeted interventions
- Cost per incremental conversion
- Reduced promotional spending on low-intent sessions
- Operational usefulness for marketing, remarketing, or customer-engagement teams

Historical model evaluation can estimate predictive performance. However, actual business uplift should be validated through a future controlled experiment, such as A/B testing.